In [ ]:
import os
from pathlib import Path

# 프로젝트 루트 설정 (jupyter를 어디서 실행하든 동작)
_cwd = Path(os.getcwd())
PROJECT_ROOT = _cwd.parent if _cwd.name == 'src' else _cwd
YOLOV5_DIR = PROJECT_ROOT / "yolov5"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"YOLOV5_DIR:   {YOLOV5_DIR}")

# YOLOv5 클론 (이미 존재하면 건너뜀)
if not YOLOV5_DIR.exists():
    os.chdir(PROJECT_ROOT)
    os.system("git clone https://github.com/ultralytics/yolov5")
else:
    print("yolov5 폴더가 이미 존재합니다. 클론을 건너뜁니다.")

In [ ]:
os.chdir(YOLOV5_DIR)
os.system("pip install -r requirements.txt")

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

# 데이터 경로 설정
data_path = str(PROJECT_ROOT / "assets" / "images" / "iris" / "CASIA-Iris-Interval")
output_path = str(YOLOV5_DIR)

# 학습용 및 검증용 폴더 생성
train_img_dir = os.path.join(output_path, 'images/train')
val_img_dir = os.path.join(output_path, 'images/val')
train_label_dir = os.path.join(output_path, 'labels/train')
val_label_dir = os.path.join(output_path, 'labels/val')

os.makedirs(train_img_dir, exist_ok=True)
os.makedirs(val_img_dir, exist_ok=True)
os.makedirs(train_label_dir, exist_ok=True)
os.makedirs(val_label_dir, exist_ok=True)

# 이미지와 라벨 파일 목록 가져오기
img_files = []
label_files = []

for root, _, files in os.walk(data_path):
    for file in files:
        if file.endswith(('.jpg', '.jpeg', '.png')):
            img_files.append(os.path.join(root, file))
            label_files.append(os.path.join(root, file + '.txt'))

# 데이터셋을 학습용과 검증용으로 분리
train_imgs, val_imgs, train_labels, val_labels = train_test_split(img_files, label_files, test_size=0.2, random_state=42)

# 파일 이동
for img, label in zip(train_imgs, train_labels):
    shutil.copy(img, train_img_dir)
    shutil.copy(label, train_label_dir)

for img, label in zip(val_imgs, val_labels):
    shutil.copy(img, val_img_dir)
    shutil.copy(label, val_label_dir)

print("데이터셋 분리 완료")

In [ ]:
# 경로 설정
train_label_dir = str(YOLOV5_DIR / "labels" / "train")
val_label_dir = str(YOLOV5_DIR / "labels" / "val")

def rename_files_in_directory(directory):
    for filename in os.listdir(directory):
        if filename.endswith(".jpg.txt"):
            # 새로운 파일 이름 생성 (".jpg" 부분 제거)
            new_filename = filename.replace(".jpg.txt", ".txt")
            old_filepath = os.path.join(directory, filename)
            new_filepath = os.path.join(directory, new_filename)

            os.rename(old_filepath, new_filepath)
            print(f"Renamed: {old_filepath} -> {new_filepath}")

rename_files_in_directory(train_label_dir)
rename_files_in_directory(val_label_dir)

print("파일 이름 변경 완료")

In [ ]:
data_yaml_content = f"""
train: {YOLOV5_DIR / "images" / "train"}
val: {YOLOV5_DIR / "images" / "val"}

nc: 1  # 클래스 개수 (홍채 클래스)
names: ['iris']
"""

data_yaml_path = str(YOLOV5_DIR / "data.yaml")

with open(data_yaml_path, 'w') as file:
    file.write(data_yaml_content)

print(f"data.yaml 생성 완료: {data_yaml_path}")

In [ ]:
os.chdir(YOLOV5_DIR)
os.system(f"python train.py --img 640 --batch 16 --epochs 100 --data {YOLOV5_DIR / 'data.yaml'} --weights yolov5s.pt")